# Лабораторная работа 2 — NLP, вариант на тройку

В этой работе используется локальный сервер Ollama и модель Qwen2.5:0.5B.  
Цель работы — отправить запросы к LLM по HTTP, получить ответы модели и сохранить результаты инференса в таблицу.

Здесь подключаются нужные библиотеки и задаются основные параметры проекта: путь для сохранения отчёта, адрес локального Ollama-сервера и название модели.  

In [16]:
from pathlib import Path
import json
from typing import List, Dict, Any

import pandas as pd
import requests

project_root = Path.cwd()
report_path = project_root / "inference_report.csv"

OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_GENERATE_URL = f"{OLLAMA_BASE_URL}/api/generate"
MODEL_NAME = "qwen2.5:0.5b"

print("Project root:", project_root)
print("Report path:", report_path)
print("Ollama URL:", OLLAMA_GENERATE_URL)
print("Model:", MODEL_NAME)

Project root: c:\Users\andrei\Desktop\mai\cfs2
Report path: c:\Users\andrei\Desktop\mai\cfs2\inference_report.csv
Ollama URL: http://localhost:11434/api/generate
Model: qwen2.5:0.5b


## Проверка подключения к Ollama

Сначала проверяем, отвечает ли локальный сервер.  

In [17]:
def check_ollama_server(base_url: str = OLLAMA_BASE_URL, timeout: int = 10) -> bool:
    """доступен ли локальный сервер Ollama по HTTP."""
    try:
        response = requests.get(f"{base_url}/api/tags", timeout=timeout)
        response.raise_for_status()
        return True
    except requests.RequestException as exc:
        print("Ошибка подключения к Ollama:", exc)
        return False


server_ok = check_ollama_server()
print("Ollama server available:", server_ok)

Ollama server available: True


###  Проверка списка локально установленных моделей


In [18]:
def list_local_models(base_url: str = OLLAMA_BASE_URL, timeout: int = 10) -> Dict[str, Any]:
    """Возвращает JSON со списком локально доступных моделей Ollama."""
    response = requests.get(f"{base_url}/api/tags", timeout=timeout)
    response.raise_for_status()
    return response.json()


if server_ok:
    models_payload = list_local_models()
    print(json.dumps(models_payload, ensure_ascii=False, indent=2))
else:
    print("Сервер недоступен, список моделей получить нельзя.")

{
  "models": [
    {
      "name": "qwen2.5:0.5b",
      "model": "qwen2.5:0.5b",
      "modified_at": "2026-04-22T23:43:58.5475843+03:00",
      "size": 397821319,
      "digest": "a8b0c51577010a279d933d14c2a8ab4b268079d44c5c8830c0a93900f1827c67",
      "details": {
        "parent_model": "",
        "format": "gguf",
        "family": "qwen2",
        "families": [
          "qwen2"
        ],
        "parameter_size": "494.03M",
        "quantization_level": "Q4_K_M"
      }
    }
  ]
}


## Один тестовый запрос

Сначала делаем один простой запрос, чтобы убедиться, что модель реально отвечает

In [19]:
def generate_text(
    prompt: str,
    model: str = MODEL_NAME,
    url: str = OLLAMA_GENERATE_URL,
    timeout: int = 120,
) -> str:
    """Отправляет один запрос в Ollama HTTP API и возвращает текст ответа модели."""
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
    }

    response = requests.post(url, json=payload, timeout=timeout)
    response.raise_for_status()

    data = response.json()
    return data.get("response", "").strip()


test_prompt = "Привет! Кратко представься."
if server_ok:
    test_response = generate_text(test_prompt)
    print("PROMPT:")
    print(test_prompt)
    print("\nRESPONSE:")
    print(test_response)
else:
    print("Сервер недоступен, тестовый запрос пропущен.")

PROMPT:
Привет! Кратко представься.

RESPONSE:
Привет! Я искусственный интеллект построенный на основе машинного обучения с учителем. Я способен понимать и отвечать на ваши вопросы в пределах допустимых границ. Давайте обсудим о чем-нибудь интересном!


## Набор из 10 запросов
По заданию нужно прогнать скрипт через 10 произвольных запросов и зафиксировать вывод модели.

In [20]:
prompts: List[str] = [
    "Привет! Кто ты?",
    "Кто такой Найтвинг?",
    "Как у тебя дела?",
    "Составь мне рацион еды на неделю",
    "Стоит ли учиться в ВУЗ МАИ?",
    "Твоё любимое государство?",
    "Напиши какую-нибудь английскую скороговорку",
    "Нравится ли тебе siri?",
    "Посоветуй сериал",
    "Объясни, что такое API, очень простыми словами"
]

print(f"Количество запросов: {len(prompts)}")
for i, prompt in enumerate(prompts, start=1):
    print(f"{i}. {prompt}")

Количество запросов: 10
1. Привет! Кто ты?
2. Кто такой Найтвинг?
3. Как у тебя дела?
4. Составь мне рацион еды на неделю
5. Стоит ли учиться в ВУЗ МАИ?
6. Твоё любимое государство?
7. Напиши какую-нибудь английскую скороговорку
8. Нравится ли тебе siri?
9. Посоветуй сериал
10. Объясни, что такое API, очень простыми словами


## Массовый прогон запросов

Эта ячейка отправляет все 10 запросов в модель и собирает результаты в таблицу.

In [21]:
def run_inference_batch(prompts_list: List[str], model: str = MODEL_NAME) -> pd.DataFrame:
    """Прогоняет список запросов через модель и возвращает таблицу с результатами."""
    rows = []

    for idx, prompt in enumerate(prompts_list, start=1):
        print(f"[{idx}/{len(prompts_list)}] Обрабатывается запрос...")
        response_text = generate_text(prompt=prompt, model=model)

        print("=" * 80)
        print(f"Запрос {idx}:")
        print(prompt)
        print("\nОтвет (первые 300 символов):")
        print(response_text[:300] + ("..." if len(response_text) > 300 else ""))
        print()

        rows.append(
            {
                "prompt_id": idx,
                "prompt": prompt,
                "response": response_text,
            }
        )

    return pd.DataFrame(rows)


if server_ok:
    results_df = run_inference_batch(prompts)
    results_df
else:
    print("Сервер недоступен, пакетный прогон пропущен.")

[1/10] Обрабатывается запрос...
Запрос 1:
Привет! Кто ты?

Ответ (первые 300 символов):
Hello! I'm Qwen, an artificial intelligence language model created by Alibaba Cloud. I can answer questions, assist with tasks, and provide information on various topics. How may I assist you today?

[2/10] Обрабатывается запрос...
Запрос 2:
Кто такой Найтвинг?

Ответ (первые 300 символов):
Извините, но я не могу предоставить точную информацию о конкретном персонаже "Найтвинг". Информация по этому персонажу может быть неоднозначной или недоступной. Возможно, вы имеете в виду:

1. Историческое сюжет "Найтвинг" (2022), который показался мне как "Найтвин", что является неправильным и може...

[3/10] Обрабатывается запрос...
Запрос 3:
Как у тебя дела?

Ответ (первые 300 символов):
У меня дела не такими подробными. Моя основная функция - помогать пользователям с информацией и задачами. Я умею ответить на вопросы, выполнять задачи и общаться с пользователями в различных языках. Я полезен не только для тех

## Сохранение отчёта инференса

По заданию нужен отчёт с двумя столбцами:
- запрос к LLM
- соответствующий вывод LLM

создаётся CSV-файл `inference_report.csv`.

In [22]:
def save_inference_report(df: pd.DataFrame, output_path: Path = report_path) -> None:
    """Сохраняет таблицу с результатами инференса в CSV-файл."""
    export_df = df[["prompt", "response"]].copy()
    export_df.to_csv(output_path, index=False, encoding="utf-8-sig")


if server_ok:
    save_inference_report(results_df)
    print("Отчёт сохранён в:", report_path)
else:
    print("Сервер недоступен, отчёт не сохранён.")

Отчёт сохранён в: c:\Users\andrei\Desktop\mai\cfs2\inference_report.csv


## Просмотр итоговой таблицы

Здесь выводится итоговая таблица с ответами модели

In [23]:
if server_ok:
    final_report_df = results_df[["prompt", "response"]].copy()
    final_report_df
else:
    print("Нет данных для отображения.")


Для удобства анализа ниже показаны сами запросы и первые символы ответов модели. Полные ответы сохранены в файле `inference_report.csv`.

In [24]:
import pandas as pd

preview_df = results_df.copy()
preview_df["response_preview"] = preview_df["response"].apply(
    lambda x: x[:300] + ("..." if len(x) > 300 else "")
)

preview_df[["prompt_id", "prompt", "response_preview"]]

,prompt_id,prompt,response_preview
0,1,Привет! Кто ты?,"Hello! I'm Qwen, an artificial intelligence la..."
1,2,Кто такой Найтвинг?,"Извините, но я не могу предоставить точную инф..."
2,3,Как у тебя дела?,У меня дела не такими подробными. Моя основная...
3,4,Составь мне рацион еды на неделю,Конечно! Давайте составим рацион еды для 14 дн...
4,5,Стоит ли учиться в ВУЗ МАИ?,"Извините за путаницу, но как искусственный инт..."
5,6,Твоё любимое государство?,"Извините, но я, Qwen, - не могу иметь уважител..."
6,7,Напиши какую-нибудь английскую скороговорку,Конечно! Вот пример английской скороговорки:\n...
7,8,Нравится ли тебе siri?,"Извините, я не могу отвечать на этот вопрос."
8,9,Посоветуй сериал,"Конечно, я с удовольствием помогу вам выбрать ..."
9,10,"Объясни, что такое API, очень простыми словами",API — это протоколы взаимодействия между разли...


## Вывод

В ходе работы был использован локальный сервер Ollama и модель `Qwen2.5:0.5B`.  
Был реализован Python-скрипт, который отправляет HTTP-запросы к модели, получает ответы и сохраняет результаты инференса в CSV-файл.

В рамках лабораторной было выполнено тестирование модели на 10 произвольных запросах.  
Для каждого запроса был получен текстовый ответ, а итоговый отчёт сохранён в виде таблицы с двумя столбцами: запрос и ответ модели.
